# 大语言模型低比特量化

> Decode 阶段每生成一个 Token，都要不断读取模型权重。7B 模型用 BF16 权重时，仅参数就约 14 GB。
>
> 这一章只回答一个问题：**怎么用更少的 bit 保存和计算，同时尽量不把模型搞坏？**
>
> 读完后，你应该能看懂 `W4A16`、`W8A8`、FP8、GPTQ、AWQ、SmoothQuant、GGUF，以及它们分别在量化什么。

先看最直观的收益。


In [ ]:
def weight_size_gb(params_billion, bits):
    return params_billion * 1e9 * bits / 8 / 1e9

for bits in [16, 8, 4]:
    print(f"7B @ {bits:2d}-bit -> {weight_size_gb(7, bits):4.1f} GB (ideal)")


## 1. INT4 只有 16 个格子，浮点数怎么塞进去？

量化不是“把小数直接 round 成整数”。真正的问题是：

> 这 16 个格子覆盖多大的浮点范围？每个格子间隔多大？

这个间隔就是 `scale`。对称量化可以写成：

$$
q = \mathrm{clip}(\mathrm{round}(x/s), q_{min}, q_{max}),
\qquad \hat{x}=s q
$$


In [ ]:
import numpy as np

x = np.array([-1.0, -0.72, -0.31, 0.0, 0.18, 0.63, 1.0], dtype=np.float32)
qmax = 7
scale = np.max(np.abs(x)) / qmax
q = np.round(x / scale).clip(-qmax, qmax).astype(np.int32)
x_hat = q * scale

print("scale:", round(float(scale), 4))
print("float:", x)
print("INT4 :", q)
print("dequant:", np.round(x_hat, 3))
print("MAE:", np.mean(np.abs(x-x_hat)))


## 2. 为什么不能整层共用一个 scale？

真实权重里，不同 channel / group 的范围差别很大。一个 outlier 就可能把全层的 scale 拉粗。

所以你会在量化配置里看到：

- per-tensor
- per-channel
- per-group / group size 32、64、128
- per-token activation quantization

粒度越细，误差通常越小，但 scale 元数据更多、Kernel 也更复杂。


In [ ]:
np.random.seed(7)
W = np.random.randn(4, 16).astype(np.float32) * 0.25
W[1] *= 8

def qdq(a, axis=None):
    m = np.max(np.abs(a), axis=axis, keepdims=True)
    s = np.maximum(m/7, 1e-12)
    q = np.round(a/s).clip(-7,7)
    return q*s

err_tensor = np.mean(np.abs(W-qdq(W)))
err_channel = np.mean(np.abs(W-qdq(W, axis=1)))

print("per-tensor MAE :", round(float(err_tensor),4))
print("per-channel MAE:", round(float(err_channel),4))


## 3. W4A16 / W8A8 到底在说什么？

写法通常是：

```text
W4A16  = Weight 4-bit, Activation 16-bit
W8A8   = Weight 8-bit, Activation 8-bit
W4A8   = Weight 4-bit, Activation 8-bit
```

**Weight-only** 最容易理解：权重低比特保存，计算时由低精度 Kernel 或反量化参与矩阵乘。

Activation 更难量化，因为它会随输入变化，而且经常有 outlier。


## 4. GPTQ、AWQ、SmoothQuant：三个名字分别在“救”什么？

把它们记成三种不同问题就够了：

- **GPTQ**：权重量化之后误差怎么补？使用二阶信息近似，按块量化并补偿。
- **AWQ**：哪些权重最不能被量化坏？根据 Activation 统计保护更重要的通道。
- **SmoothQuant**：Activation outlier 太难量化怎么办？把一部分数值难度从 Activation 平滑转移到 Weight。

不要把它们背成三个“量化格式”。它们首先是**量化算法 / 策略**。


## 5. PTQ、QAT、FP8、KV Cache Quantization 放在哪里？

```text
PTQ  : 模型训练完之后再量化
QAT  : 训练时就模拟量化误差，让模型适应
FP8  : 浮点 8-bit，常见于现代 GPU 的训练 / 推理路径
KV quantization : 不只压权重，还压不断增长的 KV Cache
```

看到 `FP8 KV Cache` 时，不要和 `W4A16` 混在一起：它们压的是不同对象。


## 6. 真正下载模型时：GPTQ / AWQ / GGUF 怎么选？

一个实用判断：

| 你在哪里跑 | 常见选择 |
|---|---|
| NVIDIA GPU 服务 | vLLM / SGLang 支持的 AWQ、GPTQ、FP8 等 |
| CPU / Mac / 本地端侧 | GGUF + llama.cpp |
| 想自己做量化实验 | Transformers / AutoGPTQ / AWQ 等工具链 |
| 高吞吐数据中心 | 看硬件支持的 FP8 / INT8 / INT4 Kernel |

`Q4_K_M` 这类名字主要来自 GGUF / llama.cpp 生态，描述具体量化类型和分组策略；它和 `AWQ` 不是同一层概念。


In [ ]:
# 一个最小“读配置”练习
configs = {
    "W4A16": "权重 4-bit，activation 仍高精度",
    "W8A8": "权重和 activation 都 8-bit",
    "FP8 KV": "KV Cache 用 FP8 保存",
    "GGUF Q4_K_M": "GGUF 生态中的 4-bit 家族量化格式",
}
for k,v in configs.items():
    print(f"{k:<12} -> {v}")


## 小结

读到任何量化名词，先问四个问题：

```text
量化谁？      weight / activation / KV cache
量化成什么？ INT4 / INT8 / FP8 ...
粒度多细？   tensor / channel / group / token
怎么控制误差？RTN / GPTQ / AWQ / SmoothQuant / QAT ...
```

下一章继续沿着 Decode 的瓶颈走：

> **每一步便宜了，但还是一次 Target forward 只能确认一个 Token。能不能一次确认多个？**
